# 🧪 Praktikum 01 – Entwicklungsumgebung & erste LLM-Interaktion
**Applied Generative AI – NLP | Sommersemester 2026**

> ⏱️ **Gesamtdauer: ~90 Minuten**  
> 🎯 **Lernziele:** Python-Umgebung verifizieren · Ollama installieren & bedienen · lokale LLM-Inferenz via REST-API und Python-Library · Systemparameter erkunden

---

## Vorbereitung – Was du brauchst
| Tool | Version | Installationsquelle |
|------|---------|---------------------|
| Python | 3.11+ | python.org |
| Ollama | aktuell | [ollama.com](https://ollama.com) |
| Jupyter Lab/Notebook | aktuell | `pip install jupyterlab` |

**Terminal-Befehl zum Starten des Modells:**
```bash
ollama pull qwen2.5:7b
ollama serve          # Falls noch nicht gestartet
```


In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")
print("\nOptional: LLM-Konfiguration über Umgebungsvariablen")
print("  OLLAMA_BASE_URL (für extern erreichbaren Ollama-Host)")
print("  LLM_MODEL (z. B. qwen2.5:7b)")

# Für Colab typischerweise setzen (vor Setup-Zelle):
# os.environ["OLLAMA_BASE_URL"] = "https://<dein-ollama-host>"
# os.environ["LLM_MODEL"] = "qwen2.5:7b"

if IN_COLAB and "OLLAMA_BASE_URL" not in os.environ:
    print("\nHinweis: In Colab ist localhost normalerweise nicht erreichbar.")
    print("Setze OLLAMA_BASE_URL auf einen externen Ollama-Host.")

## Studentische Checkliste (Start)

1. Optional in Colab zuerst `OLLAMA_BASE_URL` und `LLM_MODEL` setzen.
2. Danach die Setup-Zelle vollständig ausführen und bei Bedarf Runtime neu starten.
3. Den ersten Test-Call ausführen; nur bei erfolgreicher Antwort mit den restlichen Teilen weitermachen.

## Teil 1 – Umgebung prüfen ⏱️ ~10 min

In [ ]:
import importlib
import os
import platform
import subprocess
import sys
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
AUTO_INSTALL_MISSING = True

REQUIRED = {
    "requests": "2.32.3",
    "ollama": "0.4.7",
    "rich": "13.9.4",
}

missing = [name for name in REQUIRED if find_spec(name) is None]
if missing:
    print("Fehlende Pakete:", ", ".join(missing))
    if AUTO_INSTALL_MISSING:
        specs = [f"{name}=={REQUIRED[name]}" for name in missing]
        print("Installiere:", ", ".join(specs))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])
        print("Installation abgeschlossen. Bei Importfehlern Runtime neu starten.")
    else:
        print("Setze AUTO_INSTALL_MISSING=True oder installiere manuell.")

print("=" * 55)
print("  SYSTEMCHECK")
print("=" * 55)
print(f"Python      : {sys.version}")
print(f"Platform    : {platform.platform()}")
print(f"Runtime     : {'Google Colab' if IN_COLAB else 'Lokal/Jupyter'}")

for pkg in REQUIRED:
    status = "✅" if find_spec(pkg) else "❌"
    print(f"  {status}  {pkg:<12}")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").strip()
MODEL = os.getenv("LLM_MODEL", "qwen2.5:7b").strip()
OLLAMA_PY_AVAILABLE = find_spec("ollama") is not None
ollama = importlib.import_module("ollama") if OLLAMA_PY_AVAILABLE else None

print(f"Ollama-Base-URL: {OLLAMA_BASE_URL}")
if IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⚠️ In Colab ist localhost meist nicht erreichbar.")
    print("   Setze OLLAMA_BASE_URL auf einen extern erreichbaren Ollama-Host.")

In [ ]:
# Fehlende Pakete installieren (falls nötig)
# !pip install requests ollama rich


## Teil 2 – Ollama API direkt mit `requests` ⏱️ ~20 min

Ollama stellt unter `http://localhost:11434` eine REST-API bereit.  
Wir nutzen zunächst das rohe HTTP-Interface, um zu verstehen, wie LLM-Kommunikation *unter der Haube* funktioniert.

### 2.1 – Verfügbare Modelle abfragen

In [ ]:
import requests
import json

resp = None
OLLAMA_API_AVAILABLE = False

try:
    resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    resp.raise_for_status()
    OLLAMA_API_AVAILABLE = True
except Exception as e:
    print("❌ Ollama-API nicht erreichbar:", e)
    print("   Lokal: ollama serve")
    print("   Colab: OLLAMA_BASE_URL auf externen Host setzen")

models = []
if OLLAMA_API_AVAILABLE and resp is not None:
    models = resp.json().get("models", [])

print(f"Lokale Modelle ({len(models)}):")
for m in models:
    size_gb = m.get("size", 0) / 1e9
    print(f"  • {m['name']:<30}  {size_gb:.1f} GB")

### 2.2 – Einfache Completion (non-streaming)

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Erkläre in einem Satz, was ein Transformer ist.",
    "stream": False,
    "options": {"temperature": 0.3}
}

if not OLLAMA_API_AVAILABLE:
    print("⏭️ Übersprungen: Keine erreichbare Ollama-API.")
else:
    resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=120)
    resp.raise_for_status()

    data = resp.json()
    print("Antwort:", data["response"])
    print()
    print(f"Tokens erzeugt   : {data.get('eval_count', '?')}")
    print(f"Tokens/Sek       : {data.get('eval_count', 0) / max(data.get('eval_duration', 1), 1) * 1e9:.1f}")
    print(f"Gesamtdauer      : {data.get('total_duration', 0) / 1e9:.2f}s")

### 2.3 – Streaming-Antwort in Echtzeit

In [ ]:
payload_stream = {
    "model": MODEL,
    "prompt": "Nenne die 5 wichtigsten Meilensteine in der KI-Geschichte.",
    "stream": True,
    "options": {"temperature": 0.5}
}

if not OLLAMA_API_AVAILABLE:
    print("⏭️ Übersprungen: Keine erreichbare Ollama-API.")
else:
    print("Streaming-Antwort:\n" + "─" * 50)
    with requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload_stream, stream=True, timeout=180) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if line:
                chunk = json.loads(line)
                print(chunk.get("response", ""), end="", flush=True)
                if chunk.get("done"):
                    break
    print("\n" + "─" * 50)

## Teil 3 – Chat-Interface mit der `ollama` Python-Library ⏱️ ~20 min

Die `ollama`-Library ist eine höhere Abstraktion über die REST-API  
und unterstützt **Multi-Turn-Conversations** mit Rollenverwaltung.

In [ ]:
if not OLLAMA_PY_AVAILABLE:
    print("⏭️ Übersprungen: Python-Paket 'ollama' ist nicht verfügbar.")
elif IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⏭️ Übersprungen: In Colab ist lokales Ollama nicht erreichbar.")
    print("   Setze OLLAMA_BASE_URL auf einen externen Host.")
else:
    ollama_mod = importlib.import_module("ollama")
    response = ollama_mod.chat(
        model=MODEL,
        messages=[
            {"role": "user", "content": "Was ist der Unterschied zwischen KI und Machine Learning?"}
        ]
    )
    msg = response.get("message", {}) if isinstance(response, dict) else getattr(response, "message", {})
    print(msg.get("content", "") if isinstance(msg, dict) else str(msg))

### 3.1 – Mehrrundigem Gespräch führen

In [ ]:
if not OLLAMA_PY_AVAILABLE:
    print("⏭️ Übersprungen: Python-Paket 'ollama' ist nicht verfügbar.")
elif IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⏭️ Übersprungen: In Colab ist lokales Ollama nicht erreichbar.")
    print("   Setze OLLAMA_BASE_URL auf einen externen Host.")
else:
    ollama_mod = importlib.import_module("ollama")
    conversation = [
        {"role": "system",  "content": "Du bist ein präziser NLP-Tutor. Antworte auf Deutsch, maximal 3 Sätze."},
        {"role": "user",    "content": "Was ist ein Large Language Model?"},
    ]

    resp = ollama_mod.chat(model=MODEL, messages=conversation)
    first_msg = resp.get("message", {}) if isinstance(resp, dict) else getattr(resp, "message", {})
    assistant_reply = first_msg.get("content", "") if isinstance(first_msg, dict) else str(first_msg)
    conversation.append({"role": "assistant", "content": assistant_reply})

    print("Assistent:", assistant_reply)

    follow_up = "Wie viele Parameter hat GPT-4 ungefähr?"
    conversation.append({"role": "user", "content": follow_up})

    resp2 = ollama_mod.chat(model=MODEL, messages=conversation)
    second_msg = resp2.get("message", {}) if isinstance(resp2, dict) else getattr(resp2, "message", {})
    print("\nFollowup-Antwort:", second_msg.get("content", "") if isinstance(second_msg, dict) else str(second_msg))

## Teil 4 – Parameter erkunden ⏱️ ~25 min

### 4.1 – Temperature-Effekt

`temperature` kontrolliert die Zufälligkeit der Ausgabe:  
- `0.0` → meist sehr stabil / oft gleich, aber nicht in jeder Umgebung strikt identisch  
- `1.0` → kreativer / variabler  
- `> 1.5` → deutlich unvorhersehbarer

In [ ]:
prompt = "Erfinde einen kreativen Namen für ein KI-Startup."

if not OLLAMA_PY_AVAILABLE:
    print("⏭️ Übersprungen: Python-Paket 'ollama' ist nicht verfügbar.")
elif IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⏭️ Übersprungen: In Colab ist lokales Ollama nicht erreichbar.")
else:
    ollama_mod = importlib.import_module("ollama")
    print("Gleicher Prompt, verschiedene Temperaturen:\n")
    for temp in [0.0, 0.5, 1.0, 1.5]:
        resp = ollama_mod.generate(model=MODEL, prompt=prompt,
                                   options={"temperature": temp, "seed": 42})
        print(f"  T={temp:.1f} → {resp['response'].strip()[:80]}")

### 4.2 – System-Prompt Einfluss demonstrieren

In [ ]:
user_question = "Erkläre mir Backpropagation."

personas = {
    "Grundschullehrer":
        "Du erklärst komplexe Themen so einfach, dass sie ein 10-jähriges Kind verstehen kann.",
    "Wissenschaftler":
        "Du gibst exakte, mathematisch präzise Antworten mit Fachbegriffen.",
    "Comedian":
        "Du erklärst alles mit Witzen und Analogien zum Alltag.",
}

if not OLLAMA_PY_AVAILABLE:
    print("⏭️ Übersprungen: Python-Paket 'ollama' ist nicht verfügbar.")
elif IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⏭️ Übersprungen: In Colab ist lokales Ollama nicht erreichbar.")
else:
    ollama_mod = importlib.import_module("ollama")
    for name, system_prompt in personas.items():
        resp = ollama_mod.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_question},
            ],
            options={"temperature": 0.7}
        )
        msg = resp.get("message", {}) if isinstance(resp, dict) else getattr(resp, "message", {})
        text = (msg.get("content", "") if isinstance(msg, dict) else str(msg)).strip().replace("\n", " ")
        print(f"[{name}]\n{text[:200]}...\n")

### 4.3 – Latenz-Benchmark ⏱️ ~10 min

In [ ]:
import time

test_prompts = [
    "Nenne die Hauptstadt von Deutschland.",
    "Erkläre in 3 Sätzen, was ein neuronales Netz ist.",
    "Schreibe eine 10-Punkte-Zusammenfassung der wichtigsten KI-Entwicklungen seit 2017.",
]

if not OLLAMA_PY_AVAILABLE:
    print("⏭️ Übersprungen: Python-Paket 'ollama' ist nicht verfügbar.")
elif IN_COLAB and "localhost" in OLLAMA_BASE_URL:
    print("⏭️ Übersprungen: In Colab ist lokales Ollama nicht erreichbar.")
else:
    ollama_mod = importlib.import_module("ollama")
    print(f"{'Prompt (gekürzt)':<55} {'Tokens':>7} {'Sek':>6} {'Tok/s':>7}")
    print("─" * 80)
    for p in test_prompts:
        t0 = time.time()
        r = ollama_mod.generate(model=MODEL, prompt=p, options={"temperature": 0.0})
        elapsed = max(time.time() - t0, 1e-6)
        tokens = r.get("eval_count", 0) if isinstance(r, dict) else 0
        print(f"{p[:52]:<55} {tokens:>7} {elapsed:>6.2f} {tokens/elapsed:>7.1f}")

## Teil 5 – Abschluss & Reflexion ⏱️ ~5 min

### ✅ Was du heute gelernt hast

| Konzept | Umsetzung |
|---------|-----------|
| Ollama API | REST-Calls mit `requests` |
| Streaming | Echtzeit-Ausgabe in Chunks |
| Chat-Memory | Multi-Turn mit Rollenverwaltung |
| Temperature | Einfluss auf Kreativität/Determinismus |
| System-Prompts | Persona-Steuerung |

### 🧩 Aufgaben zur Vertiefung

1. **Experimentiere mit einem anderen Modell** (z.B. `llama3.2:3b`) und vergleiche Qualität und Geschwindigkeit.
2. **Baue eine einfache interaktive Chat-Schleife** in Python, die auf Nutzereingaben wartet (`input()`).
3. **Finde heraus:** Ab welcher `temperature` werden die Antworten auf eine einfache Faktenfrage unzuverlässig?